# Tech Challenge Fase 2
## 04.3 — Silver Streaming

Aplica regras de qualidade, remove duplicidades e separa eventos válidos e rejeitados.

## 1. Imports e configuração

In [0]:
import json
from pyspark.sql import functions as F

CONFIG_FILE_PATH = "/Volumes/workspace/default/vol_trio_drive/projetos/fiap/tech_challenge_fase2/config/config.json"
config = json.loads(dbutils.fs.head(CONFIG_FILE_PATH))
STREAMING_PATH = config["paths"]["streaming_path"]
LOG_PATH = config["paths"]["log_path"]

BRONZE_PATH = f"{STREAMING_PATH}/bronze_streaming"
SILVER_PATH = f"{STREAMING_PATH}/silver_streaming"
CHECKPOINT_PATH = f"{STREAMING_PATH}/checkpoint/silver"
REJECTED_PATH = f"{LOG_PATH}/rejected/streaming"

valid_event_types = ["indicador_atualizado","meta_atualizada","resultado_corrigido"]

## 2. Regras de qualidade e persistência

In [0]:
# ============================================================
# LEITURA CONTROLADA DO SCHEMA
# ============================================================

df_bronze_schema = spark.read.parquet(
    BRONZE_PATH
)

bronze_schema = df_bronze_schema.schema

print("Schema carregado:")
print(bronze_schema.simpleString())


# ============================================================
# LEITURA STREAMING
# ============================================================

df_stream = (
    spark.readStream
    .schema(bronze_schema)
    .format("parquet")
    .load(BRONZE_PATH)
)


# ============================================================
# REGRAS DE QUALIDADE
# ============================================================

df_validos = (
    df_stream
    .withColumn(
        "valor_valido",
        F.col("valor").between(0.0, 100.0)
    )
    .withColumn(
        "tipo_evento_valido",
        F.col("event_type").isin(
            valid_event_types
        )
    )
    .withColumn(
        "chave_territorial_valida",
        F.col("co_uf").isNotNull()
        & F.col("co_municipio").isNotNull()
    )
    .withColumn(
        "registro_valido",
        F.col("valor_valido")
        & F.col("tipo_evento_valido")
        & F.col("chave_territorial_valida")
    )
    .withColumn(
        "_silver_processed_at",
        F.current_timestamp()
    )
    .withWatermark(
        "event_timestamp",
        "1 day"
    )
    .dropDuplicates(
        ["event_id"]
    )
    .filter(
        F.col("registro_valido")
    )
)


# ============================================================
# PERSISTÊNCIA SILVER STREAMING
# ============================================================

query = (
    df_validos
    .writeStream
    .format("parquet")
    .outputMode("append")
    .option(
        "checkpointLocation",
        CHECKPOINT_PATH
    )
    .trigger(
        availableNow=True
    )
    .start(
        SILVER_PATH
    )
)

query.awaitTermination()

print("Silver Streaming concluída.")

## 3. Rejeitados e validação

In [0]:
df_bronze = spark.read.parquet(BRONZE_PATH)

df_rejeitados = (
    df_bronze
    .withColumn("valor_valido", F.col("valor").between(0.0,100.0))
    .withColumn("tipo_evento_valido", F.col("event_type").isin(valid_event_types))
    .withColumn("chave_territorial_valida", F.col("co_uf").isNotNull() & F.col("co_municipio").isNotNull())
    .filter(~(F.col("valor_valido") & F.col("tipo_evento_valido") & F.col("chave_territorial_valida")))
)

(df_rejeitados.coalesce(1).write.mode("overwrite").format("parquet").save(REJECTED_PATH))

df_silver = spark.read.parquet(SILVER_PATH)
print("Válidos:", df_silver.count())
print("Rejeitados:", df_rejeitados.count())
display(df_silver.orderBy(F.col("event_timestamp").desc()).limit(20))